`ndarrays` can be indexed using the standard Python `x[objc]` syntax where x is the array and obj is the selection. There are differenc kinds of indexing available epending on obj: basic indexing , advanced indexing and field access.

Most of the following examples show the use of indexing when referencing data in an array.The exampl work just as well when assigning to an array. 

Note that in Python, `x[(exp1, exp2, ..., expN)]` is equivalent to `x[exp1, exp2, ..., expN]`; the latter is just syntactic sugar for the former.

In [2]:
import numpy as np

## 1. Basic Indexing


### Single Element Indexing

Single element indexing works exactly like that for the other standars Python sequencecs. Its is 0-based and accepts negative indices for indexing from the end of the array.

In [2]:
x = np.arange(10)
x[2]

np.int64(2)

It is not necessary to seperate each dimensions index into its own set of square brackets.

In [3]:
x = x.reshape((2,5))
x[1,3]

np.int64(8)

Note that if one indexes a multidimensional array with fewer indices than dimensions, one gets a subdimensional array. For example:

In [4]:
x[0]

array([0, 1, 2, 3, 4])

That is, each index specified selects the array corresponding to the rest of the dimensions selected. In the above example, choosing 0 means that the remaining dimension of length 5 is being left unspecified, and that what is returned is an array of that dimensionality and size. It must be noted that the returned array is a view, i.e., it is not a copy of the original, but points to the same values in memory as does the original array. In this case, the 1-D array at the first position (0) is returned. So using a single index on the returned array, results in a single element being returned. That is:

In [5]:
x[0][2]

np.int64(2)

So note that `x[0, 2] == x[0][2]` though the second case is more inefficient as a new temporary array is created after the first index that is subsequently indexed by `2`.

>**Note**:
>NumPy uses C-order indexing. That means that the last index usually represents the most rapidly changing memory location, unlike Fortran or IDL, where the first index represents the most rapidly changing location in memory. This difference represents a great potential for confusion.

### Slicing and Striding

Basics slicing extends pythons basics concepts of slicing to N dimensions. Basics slicing occurs when objc is a `slice` object(constructed by `start:stop:step` notation inside of brackets)an integer, or a tuple of slice objects and integers. 

All arrays generated by basic slicing are always views of the original array.

>**Note**
>NumPy slicing creates a view instead of a copy as in the case of built-in Python sequences such as string, tuple and list. Care must be taken when extracting a small portion from a large array which becomes useless after the extraction, because the small portion extracted contains a reference to the large original array whose memory will not be released until all arrays derived from it are garbage-collected. In such cases an explicit `copy()` is recommended


The standard rules of sequence slicing apply to basic slicing on a per-dimension basis (including using a step index). Some useful concepts to remember include:

1. The basic slice syntax is i:j:k where i is the starting index, j is the stopping index, and k is the step (
). This selects the m elements (in the corresponding dimension) with index values i, i + k, …, i + (m - 1) k where 
 and q and r are the quotient and remainder obtained by dividing j - i by k: j - i = q k + r, so that i + (m - 1) k < j. For example:

In [3]:
x = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
x[1:7:2]


array([1, 3, 5])

2. Negative i and j are interpreted as n + i and n + j where n is the number of elements in the corresponding dimension. Negative k makes stepping go towards smaller indices. From the above example:

In [4]:
x[-2:10]

array([8, 9])

In [5]:
x[-3:3:-1]

array([7, 6, 5, 4])

3. Assume n is the number of elements in the dimension being sliced. Then, if i is not given it defaults to 0 for k > 0 and n - 1 for k < 0 . If j is not given it defaults to n for k > 0 and -n-1 for k < 0 . If k is not given it defaults to 1. Note that `::` is the same as `:` and means select all indices along this axis. From the above example:

In [7]:
x[5:]

array([5, 6, 7, 8, 9])

4. If the number of objects in the selection tuple is less than `N`, then : is assumed for any subsequent dimensions. For example:

In [8]:
x = np.array([[[1],[2],[3]], [[4],[5],[6]]])
x.shape

(2, 3, 1)

In [9]:
x[1:2]

array([[[4],
        [5],
        [6]]])

5. An integer,` i`, returns the same values as `i:i+1` except the dimensionality of the returned object is reduced by 1. In particular, a selection tuple with the p-th element an integer (and all other entries :) returns the corresponding sub-array with dimension `N - 1`. If N = 1 then the returned object is an array scalar.
6. If the selection tuple has all entries : except the p-th entry which is a slice object `i:j:k`, then the returned array has dimension N formed by stacking, along the p-th axis, the sub-arrays returned by integer indexing of elements `i`, `i+k`, …, `i + (m - 1) k` < `j`.
7. Basic slicing with more than one non-: entry in the slicing tuple, acts like repeated application of slicing using a single non-: entry, where the non-: entries are successively taken (with all other non-: entries replaced by :). Thus, `x[ind1, ..., ind2,:]` acts like `x[ind1][..., ind2, :]` under basic slicing.
8. You may use slicing to set values in the array, but (unlike lists) you can never grow the array. The size of the value to be set in`x[obj]` = value must be (broadcastable to) the same shape as `x[obj]`.
9. A slicing tuple can always be constructed as obj and used in the `x[obj]` notation. Slice objects can be used in the construction in place of the `[start:stop:step]` notation. For example, x[1:10:5, ::-1] can also be implemented as obj = (slice(1, 10, 5), slice(None, None, -1)); x[obj] . This can be useful for constructing generic code that works on arrays of arbitrary dimensions. 

## Dimnsional Indexing tools

There are some tools to facilitate the easy matching of array shapes with expressions and in assignments.


**Ellipsis** expands to the number of `:` objects needed for the selection tuple to index all dimensions. In most cases, this means that the length of the expanded selection tuple is `x.ndim`. There may only be a single ellipsis present. From the above example:

In [10]:
x[...,0]

array([[1, 2, 3],
       [4, 5, 6]])

This is equivalent to:

In [11]:
x[:,:,0]

array([[1, 2, 3],
       [4, 5, 6]])

Each **newaxis** object in the selection tuple serves to expand the dimensions of the resulting selection by one unit-length dimension. The added dimension is the position of the **newaxis** object in the selection tuple. **newaxis** is an alias for None, and None can be used in place of this with the same result. From the above example:

In [12]:
x[:,np.newaxis,:,:].shape

(2, 1, 3, 1)

In [13]:
x[:,None, :,:].shape

(2, 1, 3, 1)

This can be handy to combine two arrays in a way that otherwise would require explicit reshaping operations. For example:

In [14]:
x = np.arange(5)
x[:, np.newaxis] + x[np.newaxis, :]

array([[0, 1, 2, 3, 4],
       [1, 2, 3, 4, 5],
       [2, 3, 4, 5, 6],
       [3, 4, 5, 6, 7],
       [4, 5, 6, 7, 8]])